# Per-Ethnicity SHAP Analysis

Compute mean |SHAP| values per ethnicity using the trained XGBoost model.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import json
from pathlib import Path

# Load data
df = pd.read_csv("../data/features.csv")
print(f"Total samples: {len(df)}")
print(f"Ethnicities: {df['ethnicity'].value_counts().to_dict()}")

In [ ]:
# Load model metadata for feature columns
with open("../models/xgboost/artifacts/metadata.json") as f:
    meta = json.load(f)

feature_cols = meta["feature_cols"]
print(f"Features: {len(feature_cols)}")

In [ ]:
# Train XGBoost on all data (same params as saved model)
from sklearn.model_selection import train_test_split

X = df[feature_cols].values
y = df["score"].values  # z-normalized score

params = meta["params"]
model = xgb.XGBRegressor(
    n_estimators=params["n_estimators"],
    max_depth=params["max_depth"],
    learning_rate=params["learning_rate"],
    subsample=params["subsample"],
    colsample_bytree=params["colsample_bytree"],
    early_stopping_rounds=params["early_stopping_rounds"],
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print(f"Model trained, best iteration: {model.best_iteration}")

In [ ]:
# Compute SHAP values for all samples
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)
print(f"SHAP values shape: {shap_values.shape}")

In [ ]:
# Per-ethnicity mean |SHAP|
shap_df = pd.DataFrame(np.abs(shap_values), columns=feature_cols)
shap_df["ethnicity"] = df["ethnicity"].values

# Compute mean |SHAP| per ethnicity
per_ethnicity = shap_df.groupby("ethnicity")[feature_cols].mean()

# Show top-5 features per ethnicity
results = []
for eth in per_ethnicity.index:
    row = per_ethnicity.loc[eth].sort_values(ascending=False)
    top5 = row.head(5)
    results.append({
        "Ethnicity": eth.capitalize(),
        "Top-5 Features": ", ".join([f.replace("_", "\_") for f in top5.index]),
        "Top SHAP": f"{top5.iloc[0]:.3f}"
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Detailed view: full ranking per ethnicity
print("\n=== Full feature ranking per ethnicity ===")
for eth in per_ethnicity.index:
    row = per_ethnicity.loc[eth].sort_values(ascending=False)
    print(f"\n--- {eth.upper()} (N={len(df[df['ethnicity']==eth])}) ---")
    for i, (feat, val) in enumerate(row.items()):
        marker = " <-- " if feat == "canthal_tilt" else ""
        print(f"  {i+1:2d}. {feat:30s} {val:.4f}{marker}")

In [ ]:
# Where does canthal_tilt rank per ethnicity?
print("\n=== Canthal tilt ranking per ethnicity ===")
for eth in per_ethnicity.index:
    row = per_ethnicity.loc[eth].sort_values(ascending=False)
    rank = list(row.index).index("canthal_tilt") + 1
    val = row["canthal_tilt"]
    print(f"  {eth:15s}: rank {rank:2d}/39, SHAP = {val:.4f}")

In [ ]:
# Save results for the report
output = {
    "per_ethnicity_top5": results,
    "per_ethnicity_mean_shap": per_ethnicity.to_dict(),
    "canthal_tilt_rank": {}
}
for eth in per_ethnicity.index:
    row = per_ethnicity.loc[eth].sort_values(ascending=False)
    rank = list(row.index).index("canthal_tilt") + 1
    output["canthal_tilt_rank"][eth] = rank

with open("../models/xgboost/artifacts/shap_per_ethnicity.json", "w") as f:
    json.dump(output, f, indent=2)

print("Saved to ../models/xgboost/artifacts/shap_per_ethnicity.json")